# Ch 02 · MuJoCo Essentials for MJWarp Users

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/02-mujoco-essentials.ipynb)

Companion notebook for the chapter page `chapters/02-mujoco-essentials.html` ([page](../chapters/02-mujoco-essentials.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Rendering notebooks only: let glvnd find the NVIDIA EGL driver (from the official
# mujoco_warp tutorial). Must run before mujoco is imported by the setup cell below.
import os
os.environ.setdefault("MUJOCO_GL", "egl")
NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
try:
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
        with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
            f.write('{"file_format_version": "1.0.0", '
                    '"ICD": {"library_path": "libEGL_nvidia.so.0"}}\n')
    print("EGL ICD file:", NVIDIA_ICD_CONFIG_PATH, "| MUJOCO_GL =", os.environ["MUJOCO_GL"])
except OSError as e:
    print("could not write the EGL ICD file (fine outside Colab):", e)

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
os.environ.setdefault("MUJOCO_GL", "egl")
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

## How to use this notebook
Most of this chapter runs **C MuJoCo** (`mujoco.mj_step`, `mujoco.MjData`) on the CPU: it teaches the vocabulary that MJWarp keeps. Examples 2.1, 2.2, 2.9 and 2.12 also touch MJWarp (`mjw.put_model`, `mjw.make_data`, `mjw.step`, the batch renderer). Every cell is the same file as on the chapter page; run them top to bottom. Examples are numbered by file (`exN_*.py`) and appear in the order of the chapter's sections, where their topic is taught.

## §2 MJCF anatomy and model sizes
`PLAYGROUND` has a ball and a box on free joints and a hinged arm. Body 0 is the world, so `nbody` is 4. The MJWarp `Model` keeps the same size fields as plain Python ints.

In [ ]:
# Example 2.1 - ex1_sizes.py
"""Example 2.1 - MJCF -> MjModel sizes, and the same sizes on the MJWarp Model.

Loads the shared PLAYGROUND model (a ball, a box, a hinged arm), prints the size fields of the
compiled mujoco.MjModel, then uploads it with mjw.put_model and checks that MJWarp keeps the same
names and values (sizes are plain Python ints on the MJWarp Model).
Run: python ex1_sizes.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/>
      <geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/>
      <geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

SIZES = ["nq", "nv", "nu", "nactuator", "na", "nbody", "njnt", "ngeom", "nsite",
         "nsensor", "nsensordata", "nmocap", "neq", "nkey", "ncam"]

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
print("MjModel sizes:")
for name in SIZES:
    print(f"  {name:12s} {getattr(mjm, name)}")

# Body 0 is always the world body; ngeom counts the floor, which belongs to the world body.
BODY = mujoco.mjtObj.mjOBJ_BODY
print("body names:", [mujoco.mj_id2name(mjm, BODY, i) for i in range(mjm.nbody)])
print("geom 0 is", mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, 0),
      "on body", mjm.geom_bodyid[0])
print("id of body 'box':", mujoco.mj_name2id(mjm, BODY, "box"))

# The MJWarp Model mirrors the MjModel size fields (plain Python ints).
m = mjw.put_model(mjm)
CHECK = ["nq", "nv", "nu", "na", "nbody", "ngeom", "nsite", "nsensordata", "nmocap", "neq"]
same = all(getattr(m, n) == getattr(mjm, n) for n in CHECK)
print("mjw.Model sizes equal MjModel sizes:", same, "| type(m.nq):", type(m.nq).__name__)
print("timestep on the MJWarp model:", m.opt.timestep.numpy(), "(a batched float32 array)")

## §3 Joints, coordinates and quaternions
Free joints own 7 qpos / 6 qvel, ball 4 / 3, slide and hinge 1 / 1. `jnt_qposadr` and `jnt_dofadr` say where each joint's slice starts.

In [ ]:
# Example 2.2 - ex2_joint_table.py
"""Example 2.2 - the joint table: type, qpos address, dof address and range for every joint.

Prints one row per joint of PLAYGROUND from mujoco.MjModel (jnt_type, jnt_qposadr, jnt_dofadr,
jnt_range), then reads the same address tables from the MJWarp Model, where jnt_type values are
mjw.JointType members.
Run: python ex2_joint_table.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/>
      <geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/>
      <geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""
# qpos / qvel slots per joint type (free 7/6, ball 4/3, slide 1/1, hinge 1/1)
JT = mjw.JointType
NQ = {JT.FREE: 7, JT.BALL: 4, JT.SLIDE: 1, JT.HINGE: 1}
NV = {JT.FREE: 6, JT.BALL: 3, JT.SLIDE: 1, JT.HINGE: 1}

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
print(f"{'joint':8s} {'type':6s} {'qposadr':>7s} {'nq':>3s} {'dofadr':>6s} {'nv':>3s}  range")
for j in range(mjm.njnt):
    name = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_JOINT, j) or f"(jnt {j})"
    jt = JT(int(mjm.jnt_type[j]))   # mjw.JointType mirrors mujoco.mjtJoint
    rng = mjm.jnt_range[j] if mjm.jnt_limited[j] else "unlimited"
    print(f"{name:8s} {jt.name:6s} {mjm.jnt_qposadr[j]:7d} {NQ[jt]:3d} "
          f"{mjm.jnt_dofadr[j]:6d} {NV[jt]:3d}  {rng}")
print("total: nq =", sum(NQ[JT(t)] for t in mjm.jnt_type),
      "| nv =", sum(NV[JT(t)] for t in mjm.jnt_type))

# Which joint owns each DOF, and which body owns it?
print("dof_jntid :", mjm.dof_jntid)
print("dof_bodyid:", mjm.dof_bodyid)

# The same tables on the MJWarp model (device arrays, read back with .numpy()).
m = mjw.put_model(mjm)
print("mjw jnt_qposadr:", m.jnt_qposadr.numpy(), "| jnt_dofadr:", m.jnt_dofadr.numpy())
print("mjw jnt_type   :", [JT(t).name for t in m.jnt_type.numpy()])

MuJoCo quaternions are **(w, x, y, z)**; the identity is (1, 0, 0, 0). SciPy's `as_quat()` returns (x, y, z, w) by default: feeding it unchanged gives a different rotation and no error.

In [ ]:
# Example 2.6 - ex6_quaternions.py
"""Example 2.6 - quaternions are (w, x, y, z) in MuJoCo, and what goes wrong if you forget.

Builds a free box, sets its orientation to 90 degrees about +z with mju_axisAngle2Quat, runs
mj_forward and reads xquat / xmat. Then it feeds the same four numbers in (x, y, z, w) order to
show the wrong rotation you get, and prints how qpos packs a free joint: 3 position + 4 quat.
Run: python ex6_quaternions.py
"""
import mujoco
import numpy as np

np.set_printoptions(precision=3, suppress=True)

FREE_BOX = """
<mujoco>
  <worldbody>
    <body name="box"><freejoint/><geom type="box" size=".2 .1 .05"/></body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(FREE_BOX)
mjd = mujoco.MjData(mjm)
print("qpos0 of a free joint (x y z | w x y z):", mjm.qpos0)

quat = np.zeros(4)
mujoco.mju_axisAngle2Quat(quat, np.array([0.0, 0.0, 1.0]), np.pi / 2)
print("90 deg about +z as (w, x, y, z):", quat)

mjd.qpos[3:7] = quat           # qpos[0:3] = position, qpos[3:7] = orientation
mujoco.mj_forward(mjm, mjd)
print("xquat of the box:", mjd.xquat[1])
R = mjd.xmat[1].reshape(3, 3)
print("body x axis in world (first column of xmat):", R[:, 0], "-> points along +y")

mat = np.zeros(9)
mujoco.mju_quat2Mat(mat, quat)
print("mju_quat2Mat gives the same matrix:", np.allclose(mat, mjd.xmat[1]))

# Rotate a vector with the quaternion: the body x axis
v = np.zeros(3)
mujoco.mju_rotVecQuat(v, np.array([1.0, 0.0, 0.0]), quat)
print("mju_rotVecQuat([1,0,0]):", v)

# Wrong: someone stored (x, y, z, w), e.g. from scipy's as_quat() default
wrong = np.array([quat[1], quat[2], quat[3], quat[0]])
mujoco.mju_normalize4(wrong)
mujoco.mju_quat2Mat(mat, wrong)
angle = 2 * np.arccos(np.clip(wrong[0], -1, 1))
print("same numbers read as (x,y,z,w) -> w =", round(wrong[0], 3),
      f"angle {np.degrees(angle):.1f} deg about", wrong[1:] / np.linalg.norm(wrong[1:]))
print("wrong matrix maps x to:", mat.reshape(3, 3)[:, 0])

# Two quaternions describe every rotation: q and -q
neg = -quat
mujoco.mju_quat2Mat(mat, neg)
print("-q gives the same rotation:", np.allclose(mat, mjd.xmat[1]))

Default classes set attributes once; joint limits are soft and add one constraint row when violated.

In [ ]:
# Example 2.8 - ex8_limits_defaults.py
"""Example 2.8 - default classes and a joint limit that the arm is pushed against.

A <default class="limb"> sets the joint damping, the range and the geom size and colour once;
two links use it. The hinge range is -30..30 degrees. Gravity plus a motor command push the arm
past +30 degrees: the limit is soft, so qpos ends slightly beyond the range and one limit row
appears in the constraint list (nefc).
Run: python ex8_limits_defaults.py
"""
import mujoco
import numpy as np

np.set_printoptions(precision=4, suppress=True)

ARM = """
<mujoco model="limited_arm">
  <compiler angle="degree"/>
  <option timestep="0.002"/>
  <default>
    <default class="limb">
      <joint type="hinge" axis="0 1 0" damping="0.5" limited="true" range="-30 30"/>
      <geom type="capsule" size=".03" rgba=".2 .6 .9 1" mass="0.5"/>
    </default>
  </default>
  <worldbody>
    <body name="upper" pos="0 0 1">
      <joint name="shoulder" class="limb"/>
      <geom class="limb" fromto="0 0 0 .4 0 0"/>
      <body name="lower" pos=".4 0 0">
        <joint name="elbow" class="limb" range="-90 90"/>
        <geom class="limb" fromto="0 0 0 .3 0 0" rgba=".9 .5 .2 1"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="lift" joint="shoulder" gear="1"/></actuator>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(ARM)
mjd = mujoco.MjData(mjm)
for j in range(mjm.njnt):
    name = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_JOINT, j)
    damping = mjm.dof_damping[mjm.jnt_dofadr[j]]   # damping is stored per DOF
    print(f"joint {name:8s} damping {damping:.2f}  limited {bool(mjm.jnt_limited[j])}"
          f"  range [deg] {np.degrees(mjm.jnt_range[j])}")
print("geom rgba from the class / overridden:", mjm.geom_rgba[0], mjm.geom_rgba[1])
print("range is stored in radians:", mjm.jnt_range[0])

# Push down: gravity rotates the arm to +angle about +y (x axis toward -z); add a motor push.
mjd.ctrl[0] = 2.0
for _ in range(1000):   # 2 s
    mujoco.mj_step(mjm, mjd)
q = np.degrees(mjd.qpos)
print(f"shoulder qpos = {q[0]:.3f} deg (limit 30): beyond by {q[0] - 30:.3f} deg")
print("nefc =", mjd.nefc, "| efc_type:",
      [mujoco.mjtConstraint(t).name for t in mjd.efc_type[:mjd.nefc]])
print(f"limit force on the shoulder (qfrc_constraint[0]) = {mjd.qfrc_constraint[0]:.3f} N m")

# Inside the range there is no row at all.
mjd.ctrl[0] = 0.0
mujoco.mj_resetData(mjm, mjd)
mujoco.mj_forward(mjm, mjd)
print("at qpos = 0: nefc =", mjd.nefc)

## §4 Contacts are soft
A resting ball sinks a fraction of a millimetre (`dist < 0`). `solref = (timeconst, dampratio)` is a spring-damper: smaller timeconst = stiffer, smaller dampratio = bouncier. Add `--dump-json PATH` to record the sweep used by the chapter's widget.

In [ ]:
# Example 2.7 - ex7_contacts_soft.py
"""Example 2.7 - contacts are soft: a resting ball sinks a little into the floor.

Drops BALL_DROP, lets it settle for 1 s and prints the contact MuJoCo found: the geom pair,
dist (negative = penetration), the normal frame[0:3] and the constraint rows. Then it halves the
solref time constant (0.02 -> 0.01, a stiffer contact) and shows the smaller penetration.
Run: python ex7_contacts_soft.py [--dump-json PATH]
  --dump-json PATH  writes the penetration for a sweep of timeconst values (chapter slider, Rule D)
"""
import argparse
import json

import mujoco
import numpy as np

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()


def settle(timeconst=None, dampratio=1.0, seconds=1.0):
    """Drop the ball and return (MjModel, MjData) after `seconds` of simulation."""
    mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
    if timeconst is not None:
        mjm.geom_solref[:] = [timeconst, dampratio]   # both geoms; MuJoCo mixes the pair
    mjd = mujoco.MjData(mjm)
    for _ in range(int(seconds / mjm.opt.timestep)):
        mujoco.mj_step(mjm, mjd)
    return mjm, mjd


mjm, mjd = settle()
print("default geom solref (timeconst, dampratio):", mjm.geom_solref[1])
print("default geom solimp:", mjm.geom_solimp[1])
print("ncon =", mjd.ncon, "| nefc =", mjd.nefc, "(pyramidal cone, condim 3 -> 4 rows per contact)")
c = mjd.contact[0]
g1 = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, c.geom[0])
g2 = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, c.geom[1])
print(f"contact 0: geoms ({g1}, {g2})  dist = {c.dist:.5f} m  normal = {c.frame[:3]}")
print(f"ball z = {mjd.qpos[2]:.5f} (radius 0.1, so it sits {0.1 - mjd.qpos[2]:.5f} m deep)")

mjm2, mjd2 = settle(timeconst=0.01)
print(f"solref timeconst 0.01: dist = {mjd2.contact[0].dist:.5f} m  "
      f"(ratio {mjd2.contact[0].dist / c.dist:.2f})")

if args.dump_json:
    # For each (timeconst, dampratio): the ball height over the first 1.2 s (every 6th step,
    # 100 samples) and the resting penetration after 3 s.
    out = {"series_dt": 6 * float(mjm.opt.timestep), "runs": []}
    for tc in [0.005, 0.01, 0.02, 0.04, 0.08]:
        for dr in [0.3, 1.0]:
            m3 = mujoco.MjModel.from_xml_string(BALL_DROP)
            m3.geom_solref[:] = [tc, dr]
            d3 = mujoco.MjData(m3)
            zs = []
            for i in range(1500):   # 3 s
                mujoco.mj_step(m3, d3)
                if i < 600 and i % 6 == 0:
                    zs.append(round(float(d3.qpos[2]), 4))
            out["runs"].append({"timeconst": tc, "dampratio": dr, "z": zs,
                                "rest_dist": round(float(d3.qpos[2]) - 0.1, 6)})
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(out["runs"]), "runs")

## §5-§6 Equality constraints and mocap bodies
A mocap target circles; a weld equality drags a free box after it; `eq_active[0] = 0` releases the box. The same schedule then runs in MJWarp with `d.mocap_pos` and `d.eq_active`.

In [ ]:
# Example 2.9 - ex9_equality_mocap.py
"""Example 2.9 - a mocap target, a weld equality, and switching the weld off with eq_active.

MOCAP_TARGET: a mocap body "target" (moved by you, not by physics) and a free box welded to it.
For 2 s we move mocap_pos around a circle and the box follows; then eq_active[0] = 0 turns the
weld off and the box falls to the floor. The same schedule then runs in MJWarp, which uses the
same field names (d.mocap_pos, d.eq_active), and we compare the two trajectories.
Run: python ex9_equality_mocap.py [--dump-json PATH]
  --dump-json PATH  writes the MJWarp trajectory (target and box xyz) for the chapter replay
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

MOCAP_TARGET = """
<mujoco model="mocap_target">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="target" mocap="true" pos="0.2 0 0.5">
      <geom type="sphere" size=".03" contype="0" conaffinity="0" rgba="0 1 0 .5"/>
    </body>
    <body name="box" pos="0.2 0 0.5">
      <freejoint/>
      <geom type="box" size=".05 .05 .05" mass="0.5"/>
    </body>
  </worldbody>
  <equality><weld body1="box" body2="target" solref="0.02 1"/></equality>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

T_CIRCLE, T_FALL, R, Z = 2.0, 1.0, 0.2, 0.5


def target_at(t):
    """Target position at time t: a horizontal circle of radius R at height Z."""
    a = 2 * np.pi * t / T_CIRCLE
    return np.array([R * np.cos(a), R * np.sin(a), Z])


mjm = mujoco.MjModel.from_xml_string(MOCAP_TARGET)
mjd = mujoco.MjData(mjm)
box = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "box")
print("nmocap =", mjm.nmocap, "| neq =", mjm.neq, "| eq_type:", mujoco.mjtEq(mjm.eq_type[0]).name,
      "| body_mocapid:", mjm.body_mocapid)
nstep = int((T_CIRCLE + T_FALL) / mjm.opt.timestep)
n_on = int(T_CIRCLE / mjm.opt.timestep)
c_box = np.zeros((nstep, 3))
for i in range(nstep):
    if i == n_on:
        mjd.eq_active[0] = 0          # weld off: the box is now a free body
        print(f"t={mjd.time:.2f}  eq_active[0] = 0")
    mjd.mocap_pos[0] = target_at(mjd.time)
    mujoco.mj_step(mjm, mjd)
    c_box[i] = mjd.xpos[box]
    if (i + 1) % 250 == 0:
        gap = np.linalg.norm(mjd.xpos[box] - mjd.mocap_pos[0])
        print(f"t={mjd.time:.2f}  target={mjd.mocap_pos[0]}  box={mjd.xpos[box]}  gap={gap:.3f}")

# The same loop in MJWarp: same field names, but arrays have a leading nworld axis.
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=8, njmax=64)
w_box, w_tgt = np.zeros((nstep, 3)), np.zeros((nstep, 3))
for i in range(nstep):
    if i == n_on:
        d.eq_active.assign(np.zeros((1, mjm.neq), dtype=bool))
    t = i * mjm.opt.timestep
    d.mocap_pos.assign(target_at(t).reshape(1, 1, 3).astype(np.float32))
    mjw.step(m, d)
    w_box[i] = d.xpos.numpy()[0, box]     # host read every step: fine for 1 world, slow at scale
    w_tgt[i] = target_at(t)
print("MJWarp d.mocap_pos shape:", d.mocap_pos.shape, "| d.eq_active shape:", d.eq_active.shape)
print(f"max |box_C - box_MJWarp| while welded: {np.abs(c_box[:n_on] - w_box[:n_on]).max():.2e} m")
print(f"box z at the end: C {c_box[-1, 2]:.4f}  MJWarp {w_box[-1, 2]:.4f} (half-size 0.05)")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    k = 25   # every 25th step = 0.05 s -> 60 frames
    frames = [[[round(float(v), 3) for v in w_tgt[i]], [round(float(v), 3) for v in w_box[i]]]
              for i in range(0, nstep, k)]
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": 1, "dt": k * float(mjm.opt.timestep), "kind": "mocap3d",
                   "labels": ["target", "box"], "weld_off_frame": n_on // k,
                   "frames": frames}, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## §7 Actuators, controls and sensors
`ctrl` drives the motor (clamped by `ctrlrange`); sensor `i` lives at `sensordata[sensor_adr[i] : sensor_adr[i] + sensor_dim[i]]`.

In [ ]:
# Example 2.5 - ex5_ctrl_sensor.py
"""Example 2.5 - set ctrl, read sensordata by address, on the double PENDULUM.

A constant motor command ctrl = 2 (N m, gear 1) drives the shoulder hinge. The three sensors
(jointpos q1, jointvel v1, framepos tip_pos) live side by side in one flat sensordata vector;
sensor_adr and sensor_dim say where each one starts and how long it is.
Run: python ex5_ctrl_sensor.py
"""
import mujoco
import numpy as np

np.set_printoptions(precision=3, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(PENDULUM)
mjd = mujoco.MjData(mjm)
print("nu =", mjm.nu, "| nactuator =", mjm.nactuator, "| nsensor =", mjm.nsensor,
      "| nsensordata =", mjm.nsensordata)

slices = {}
for i in range(mjm.nsensor):
    name = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_SENSOR, i)
    adr, dim = int(mjm.sensor_adr[i]), int(mjm.sensor_dim[i])
    slices[name] = slice(adr, adr + dim)
    print(f"  sensor {name:8s} adr {adr} dim {dim}")

mjd.ctrl[0] = 2.0           # constant torque on hinge1; ctrlrange clamps to [-5, 5]
for step in range(1, 301):  # 3 s
    mujoco.mj_step(mjm, mjd)
    if step % 50 == 0:
        s = mjd.sensordata
        print(f"t={mjd.time:.1f}  q1={s[slices['q1']][0]:7.3f}  v1={s[slices['v1']][0]:7.3f}"
              f"  tip_pos={s[slices['tip_pos']]}  E={mjd.energy.sum():7.3f}")

# The framepos sensor equals the site position computed by kinematics.
same = np.allclose(mjd.site_xpos[0], mjd.sensordata[slices["tip_pos"]])
print("tip site_xpos:", mjd.site_xpos[0], "== sensor:", same)
mjd.ctrl[0] = 50.0
mujoco.mj_step(mjm, mjd)
print("ctrl=50 is clamped by ctrlrange: actuator_force =", mjd.actuator_force)

## §9 The simulation loop
`mj_step` = `mj_forward` + integrate. Record a trajectory, then reset with keyframes. On Colab, matplotlib is installed, so Example 2.3 also saves `ex3_trajectory.png`.

In [ ]:
# Example 2.3 - ex3_trajectory.py
"""Example 2.3 - step PLAYGROUND with mujoco.mj_step and record a trajectory.

Steps 200 times (2 s at timestep 0.01), records the ball and box heights (qpos z slots) and the
hinge angle, prints where they settle, then saves a plot (Rule O: numbers first, plot optional).
Run: python ex3_trajectory.py [--dump-json PATH] [--png PATH]
  --dump-json PATH  writes the recorded heights for the chapter figure (Rule D)
  --png PATH        where to save the matplotlib plot (default ex3_trajectory.png)
"""
import argparse
import json

import mujoco
import numpy as np

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/>
      <geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/>
      <geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""
parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
parser.add_argument("--png", default="ex3_trajectory.png")
args, _ = parser.parse_known_args()

NSTEP = 200
mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
mjd = mujoco.MjData(mjm)

# qpos layout (Example 2.2): ball 0..6, box 7..13, hinge 14. z is the 3rd slot of a free joint.
BALL_Z, BOX_Z, HINGE = 2, 7 + 2, 14
t, ball_z, box_z, hinge = [], [], [], []
for _ in range(NSTEP):
    mujoco.mj_step(mjm, mjd)
    t.append(mjd.time)
    ball_z.append(mjd.qpos[BALL_Z])
    box_z.append(mjd.qpos[BOX_Z])
    hinge.append(mjd.qpos[HINGE])

print(f"t = {mjd.time:.2f} s after {NSTEP} steps")
print(f"ball z: start 1.000, settled {ball_z[-1]:.4f}  (radius 0.1)")
print(f"box  z: start 1.000, settled {box_z[-1]:.4f}  (half-size 0.1)")
print(f"hinge angle at the end: {hinge[-1]:.3f} rad (no limit, no damping: it keeps swinging)")
print("contacts at the end:", mjd.ncon, "| constraint rows:", mjd.nefc)

if args.dump_json:
    keep = slice(0, NSTEP, 2)   # 100 samples
    out = {"dt": 2 * mjm.opt.timestep, "t": [round(v, 3) for v in t[keep]],
           "ball_z": [round(float(v), 4) for v in ball_z[keep]],
           "box_z": [round(float(v), 4) for v in box_z[keep]],
           "hinge": [round(float(v), 3) for v in hinge[keep]]}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    print("skipped plot: matplotlib not installed")
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(t, ball_z, label="ball z")
    ax.plot(t, box_z, label="box z")
    ax.set_xlabel("time [s]")
    ax.set_ylabel("height [m]")
    ax.legend()
    fig.tight_layout()
    fig.savefig(args.png, dpi=80)
    print("saved", args.png)

In [ ]:
# Show the saved plot (Colab has mediapy installed by the setup cell)
import os
if os.path.exists("ex3_trajectory.png"):
    import mediapy as media
    media.show_image(media.read_image("ex3_trajectory.png"))

In [ ]:
# Example 2.4 - ex4_keyframe.py
"""Example 2.4 - keyframes and resets: mj_resetDataKeyframe vs mj_resetData.

Steps PLAYGROUND for 1 s, then resets to keyframe 0 ("up": both bodies lifted to z = 2, hinge at
0.3 rad) and finally back to the default state qpos0. Prints qpos and time at each stage.
Run: python ex4_keyframe.py
"""
import mujoco
import numpy as np

np.set_printoptions(precision=3, suppress=True, linewidth=120)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/>
      <geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/>
      <geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
mjd = mujoco.MjData(mjm)

for _ in range(100):
    mujoco.mj_step(mjm, mjd)
print(f"after 1 s     t={mjd.time:.2f}  qpos={mjd.qpos}")

key = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_KEY, "up")
print("keyframe 'up' has id", key, "and stores", mjm.key_qpos.shape[1], "qpos values")
mujoco.mj_resetDataKeyframe(mjm, mjd, key)
print(f"keyframe 'up' t={mjd.time:.2f}  qpos={mjd.qpos}")
print("qvel after keyframe reset is zero:", not mjd.qvel.any())

mujoco.mj_resetData(mjm, mjd)
print(f"mj_resetData  t={mjd.time:.2f}  qpos={mjd.qpos}")
print("qpos0 (the model's default pose):", mjm.qpos0)

# A reset writes the state only. Derived quantities such as xpos are not recomputed (here
# they are zero) until mj_forward runs.
print("ball xpos before mj_forward:", mjd.xpos[1])
mujoco.mj_forward(mjm, mjd)
print("ball xpos after  mj_forward:", mjd.xpos[1])

## §10 Derived quantities and the mass matrix
MuJoCo 4 stores the mass matrix in `mjd.M` (not `qM`); `mujoco.mj_fullM(m, d, dst)` expands it.

In [ ]:
# Example 2.11 - ex11_mass_matrix.py
"""Example 2.11 - the joint-space mass matrix M of the double PENDULUM with mujoco.mj_fullM.

MuJoCo stores M in a compact sparse form (MuJoCo 4: the MjData field is M, not qM), and
mj_fullM(m, d, dst) expands it into a dense nv x nv matrix. M depends on the pose: we print it
with the arm straight (hinge2 = 0) and bent (hinge2 = 90 deg). Chapter 11 derives these numbers.
Run: python ex11_mass_matrix.py
"""
import mujoco
import numpy as np

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(PENDULUM)
mjd = mujoco.MjData(mjm)
print("nv =", mjm.nv, "| nM (stored entries) =", mjm.nM, "| MjData has M:", hasattr(mjd, "M"),
      "| has qM:", hasattr(mjd, "qM"))

dense = np.zeros((mjm.nv, mjm.nv))
for q2 in [0.0, np.pi / 2]:
    mjd.qpos[:] = [0.0, q2]
    mujoco.mj_forward(mjm, mjd)          # computes M (via the composite rigid body algorithm)
    mujoco.mj_fullM(mjm, mjd, dense)   # MuJoCo 4 signature: (m, d, dst)
    print(f"hinge2 = {np.degrees(q2):4.0f} deg  M =\n{dense}")
    print("  symmetric:", np.allclose(dense, dense.T),
          "| positive definite:", bool(np.all(np.linalg.eigvalsh(dense) > 0)))

# Kinetic energy = 0.5 * qvel^T M qvel (energy flag is enabled in this model)
mjd.qvel[:] = [1.0, -0.5]
mujoco.mj_forward(mjm, mjd)
mujoco.mj_fullM(mjm, mjd, dense)   # MuJoCo 4 signature: (m, d, dst)
ke = 0.5 * mjd.qvel @ dense @ mjd.qvel
print(f"0.5 v^T M v = {ke:.4f}  | mjd.energy[1] (kinetic) = {mjd.energy[1]:.4f}")

## §11 Building models with MjSpec

In [ ]:
# Example 2.10 - ex10_mjspec.py
"""Example 2.10 - build a model in Python with mujoco.MjSpec instead of writing XML.

box_stack(n) builds the shared box-stack scene as an MJCF string; box_stack_spec(n) builds the
same scene with MjSpec (add_body, add_freejoint, add_geom, compile). Both compile to models with
the same sizes. spec.to_xml() turns the spec back into MJCF.
Run: python ex10_mjspec.py
"""
import mujoco
import numpy as np


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


def box_stack_spec(n=5, size=0.1):
    spec = mujoco.MjSpec()
    spec.modelname = "box_stack"
    spec.option.timestep = 0.005
    spec.worldbody.add_light(pos=[0, 0, 3], dir=[0, 0, -1])
    spec.worldbody.add_geom(name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE, size=[5, 5, 0.1])
    for i in range(n):
        body = spec.worldbody.add_body(name=f"box{i}", pos=[0, 0, size + 2 * size * i + 0.001 * i])
        body.add_freejoint()
        body.add_geom(type=mujoco.mjtGeom.mjGEOM_BOX, size=[size, size, size])
    return spec


def sizes(mjm):
    return {k: getattr(mjm, k) for k in ["nq", "nv", "nbody", "ngeom", "njnt"]}


xml_model = mujoco.MjModel.from_xml_string(box_stack(3))
spec = box_stack_spec(3)
spec_model = spec.compile()
print("from XML   :", sizes(xml_model))
print("from MjSpec:", sizes(spec_model))
print("same sizes:", sizes(xml_model) == sizes(spec_model),
      "| same body_pos:", np.allclose(xml_model.body_pos, spec_model.body_pos))

# Edit the spec and recompile: a 4th box, now with its own colour
top = spec.worldbody.add_body(name="box3", pos=[0, 0, 0.1 + 0.6 + 0.003])
top.add_freejoint()
top.add_geom(type=mujoco.mjtGeom.mjGEOM_BOX, size=[0.1, 0.1, 0.1], rgba=[1, 0.3, 0.3, 1])
print("after adding box3:", sizes(spec.compile()))

xml = spec.to_xml()
print("spec.to_xml()[:200]:")
print(xml[:200])

## §12 Seeing a frame
The EGL cell at the top sets `MUJOCO_GL=egl` and writes the NVIDIA EGL ICD file, which `mujoco.Renderer` needs on a Colab GPU runtime. If the renderer still cannot start, the example falls back to the MJWarp batch renderer and says so in its output.

In [ ]:
# Example 2.12 - ex12_render.py
"""Example 2.12 - see a frame: mujoco.Renderer, with the MJWarp batch renderer as fallback.

MUJOCO_GL must be set before mujoco is imported (EGL = headless GPU OpenGL on Colab). We step
PLAYGROUND for 0.3 s and render the "overview" camera. If mujoco.Renderer cannot start (no
OpenGL/EGL, e.g. a CPU box without a display), we fall back (Rule O) to MJWarp's own ray-traced
batch renderer at a small resolution, which runs on Warp's CPU device too.
Run: python ex12_render.py [--dump-png DIR]
  --dump-png DIR  writes the frame as DIR/ex12_frame.png (the chapter embeds it, Rule D)
"""
import argparse
import os

os.environ.setdefault("MUJOCO_GL", "egl")   # before "import mujoco"

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/>
      <geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/>
      <geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""
parser = argparse.ArgumentParser()
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
mjd = mujoco.MjData(mjm)
for _ in range(30):
    mujoco.mj_step(mjm, mjd)
cam = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_CAMERA, "overview")

image, source = None, None
try:
    renderer = mujoco.Renderer(mjm, height=240, width=320)
    renderer.update_scene(mjd, camera="overview")
    image, source = renderer.render(), "mujoco.Renderer (OpenGL)"
except Exception as e:  # ImportError without PyOpenGL/EGL, RuntimeError without a GL context
    print(f"mujoco.Renderer unavailable ({type(e).__name__}); using the MJWarp batch renderer")

if image is None:
    W, H = 192, 144
    m = mjw.put_model(mjm)
    d = mjw.put_data(mjm, mjd, nworld=1, nconmax=16, njmax=64)
    rc = mjw.create_render_context(mjm, nworld=1, cam_res=(W, H), render_rgb=True,
                                   render_depth=False, render_seg=False)
    mjw.forward(m, d)
    print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())
    mjw.refit_bvh(m, d, rc)
    mjw.render(m, d, rc)
    rgb = wp.zeros((1, H, W), dtype=wp.vec3)
    mjw.get_rgb(rc, camera_index=cam, rgb_out=rgb)
    image = (np.clip(rgb.numpy()[0], 0, 1) * 255).astype(np.uint8)
    source = "MJWarp batch renderer (ray tracing)"

print("camera 'overview' id:", cam, "| image shape:", image.shape, "| dtype:", image.dtype)
print("rendered with:", source, f"| mean pixel value {image.mean():.1f}")



def save_png(path, rgb):
    """Write an (H, W, 3) uint8 image as PNG: PIL if installed, else a tiny zlib writer."""
    try:
        from PIL import Image
        Image.fromarray(rgb).save(path)
        return "PIL"
    except ImportError:
        import struct
        import zlib
        h, w, _ = rgb.shape
        raw = b"".join(b"\x00" + rgb[y].tobytes() for y in range(h))

        def chunk(tag, data):
            body = tag + data
            return struct.pack(">I", len(data)) + body + struct.pack(">I", zlib.crc32(body))
        png = (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
               + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))
        with open(path, "wb") as f:
            f.write(png)
        return "zlib"


if args.dump_png:
    os.makedirs(args.dump_png, exist_ok=True)
out = os.path.join(args.dump_png or ".", "ex12_frame.png")
print("saved", out, "with", save_png(out, np.ascontiguousarray(image)))

In [ ]:
import os
if os.path.exists("ex12_frame.png"):
    import mediapy as media
    media.show_image(media.read_image("ex12_frame.png"))

## Exercise 2.1: count the coordinates
Two free bodies and a chain of three hinges. Predict `nq` and `nv`. Fill in your answer, then uncomment the last two lines to check it.

In [ ]:
import mujoco

XML = """
<mujoco>
  <worldbody>
    <body name="a" pos="0 0 1"><freejoint/><geom type="sphere" size=".1"/></body>
    <body name="b" pos="1 0 1"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
    <body name="l1" pos="2 0 2">
      <joint type="hinge" axis="0 1 0"/><geom type="capsule" fromto="0 0 0 0 0 -.3" size=".02"/>
      <body name="l2" pos="0 0 -.3">
        <joint type="hinge" axis="0 1 0"/><geom type="capsule" fromto="0 0 0 0 0 -.3" size=".02"/>
        <body name="l3" pos="0 0 -.3">
          <joint type="hinge" axis="0 1 0"/><geom type="capsule" fromto="0 0 0 0 0 -.3" size=".02"/>
        </body>
      </body>
    </body>
  </worldbody>
</mujoco>
"""
nq_pred = ...  # your prediction
nv_pred = ...
mjm = mujoco.MjModel.from_xml_string(XML)
# assert (mjm.nq, mjm.nv) == (nq_pred, nv_pred), (mjm.nq, mjm.nv)
# print("Exercise 2.1 passed:", mjm.nq, mjm.nv)

In [ ]:
#@title Exercise 2.1: solution (double-click to show the code)
nq_pred = 2 * 7 + 3 * 1   # 17
nv_pred = 2 * 6 + 3 * 1   # 15
mjm = mujoco.MjModel.from_xml_string(XML)
assert (mjm.nq, mjm.nv) == (nq_pred, nv_pred), (mjm.nq, mjm.nv)
print("Exercise 2.1 passed:", mjm.nq, mjm.nv)

## Exercise 2.2: build your own model with MjSpec
Two hinged links (shoulder, elbow), one motor on the shoulder, sensors producing exactly 3 numbers. No XML allowed. Fill in your answer, then uncomment the last three lines to check it.

In [ ]:
import mujoco

spec = mujoco.MjSpec()
# TODO: upper body + hinge "shoulder", lower body + hinge "elbow", a motor on "shoulder",
#       sensors with 3 numbers in total
# mjm = spec.compile()
# assert mjm.nq == 2 and mjm.nu == 1 and mjm.nsensordata == 3, (mjm.nq, mjm.nu, mjm.nsensordata)
# print("Exercise 2.2 passed")

In [ ]:
#@title Exercise 2.2: solution (double-click to show the code)
import mujoco

spec = mujoco.MjSpec()
upper = spec.worldbody.add_body(name="upper", pos=[0, 0, 1])
upper.add_joint(name="shoulder", type=mujoco.mjtJoint.mjJNT_HINGE, axis=[0, 1, 0])
upper.add_geom(type=mujoco.mjtGeom.mjGEOM_CAPSULE, fromto=[0, 0, 0, 0.4, 0, 0], size=[0.03, 0, 0])
lower = upper.add_body(name="lower", pos=[0.4, 0, 0])
lower.add_joint(name="elbow", type=mujoco.mjtJoint.mjJNT_HINGE, axis=[0, 1, 0])
lower.add_geom(type=mujoco.mjtGeom.mjGEOM_CAPSULE, fromto=[0, 0, 0, 0.3, 0, 0], size=[0.03, 0, 0])
lower.add_site(name="tip", pos=[0.3, 0, 0])
spec.add_actuator(name="motor", target="shoulder", trntype=mujoco.mjtTrn.mjTRN_JOINT)
spec.add_sensor(name="tip_pos", type=mujoco.mjtSensor.mjSENS_FRAMEPOS,
                objtype=mujoco.mjtObj.mjOBJ_SITE, objname="tip")
mjm = spec.compile()
assert mjm.nq == 2 and mjm.nu == 1 and mjm.nsensordata == 3
d = mujoco.MjData(mjm)
mujoco.mj_forward(mjm, d)
print("Exercise 2.2 passed; tip at", d.sensordata)

## Stretch: close a four-bar linkage
Crank hinged at (0, 0, 1), coupler hinged to the crank's end, rocker hinged at (0.5, 0, 1); close the loop with a `<connect>` equality. Spin the crank and measure the gap between the two ends; tighten `solref` and measure again. Bonus: build it with MjSpec and step it in MJWarp.

In [ ]:
# your four-bar linkage here

In [ ]:
#@title Stretch: solution (double-click to show the code)
import mujoco
import numpy as np

FOURBAR = """
<mujoco>
  <option timestep="0.002"/>
  <default><geom contype="0" conaffinity="0"/></default>
  <worldbody>
    <body name="crank" pos="0 0 1">
      <joint name="j1" type="hinge" axis="0 1 0" damping="0.05"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.3" size=".02"/>
      <body name="coupler" pos="0 0 -.3">
        <joint name="j2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 .5 0 0" size=".02"/>
      </body>
    </body>
    <body name="rocker" pos=".5 0 1">
      <joint name="j3" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.3" size=".02"/>
    </body>
  </worldbody>
  <equality><connect body1="coupler" body2="rocker" anchor=".5 0 0"/></equality>
</mujoco>
"""
m = mujoco.MjModel.from_xml_string(FOURBAR)
d = mujoco.MjData(m)
d.qvel[0] = 2.0
gaps = []
for _ in range(1000):
    mujoco.mj_step(m, d)
    a = d.xpos[2] + d.xmat[2].reshape(3, 3) @ np.array([0.5, 0, 0])
    b = d.xpos[3] + d.xmat[3].reshape(3, 3) @ np.array([0, 0, -0.3])
    gaps.append(np.linalg.norm(a - b))
print("nq", m.nq, "neq", m.neq, "nefc", d.nefc, f"max gap {max(gaps):.4f} m")

# tighten the equality: timeconst 0.02 -> 0.005 (keep it >= 2 x timestep)
m2 = mujoco.MjModel.from_xml_string(FOURBAR.replace('anchor=".5 0 0"', 'anchor=".5 0 0" solref="0.005 1"'))
d2 = mujoco.MjData(m2)
d2.qvel[0] = 2.0
gaps2 = []
for _ in range(1000):
    mujoco.mj_step(m2, d2)
    a = d2.xpos[2] + d2.xmat[2].reshape(3, 3) @ np.array([0.5, 0, 0])
    b = d2.xpos[3] + d2.xmat[3].reshape(3, 3) @ np.array([0, 0, -0.3])
    gaps2.append(np.linalg.norm(a - b))
print(f"solref 0.005: max gap {max(gaps2):.5f} m (default about 0.0068 m)")

## Where next
Ch 3 explains the physics behind contacts, joints and integration from scratch; Ch 5 runs the first full MJWarp simulation in many worlds and shows both ways of rendering it.